In [ ]:
from  COSMOtherm.Configfiles import Setup
from COSMOtherm.funcs import gen_LIQEX_inp_file, sort_solvents_df
from doepy import build
import pyDOE3
import subprocess
import pandas as pd
import os

solvents = pd.read_csv(Setup.solvents_filepath)


temperature_range = [20, 40] # [°C] Temperature range in Celsius; From Nina [20°C, 30°C, 40°C]
massc1_lacticacid_range = [5, 20] # [g/L] Concentration of lactic acid in Fermentation Broth: From Nina [5g/L, 10g/L, 20g/L] concentration incrments in literature [50g/L, 100g/L, 150g/L, 200g/L, 250g/L]

### Conversion of mass concentration to molar fraction
M_h2o = 18.01528 # [g/mol] Molar mass of water
M_lacticacid = 90.078 # [g/mol] Molar mass of lactic acid

rho_lacticacid = 1.209 # [g/mL] Density of lactic acid
rho_h2o = 1.0 # [g/mL] Density of water

c_p_h2o = 55.5 # [mol/L] Concentration of pure water at 25°C: 
c_p_lacticacid = rho_lacticacid / M_lacticacid*(10**3) # [mol/L] Concentration of pure lactic acid at 25°C:

c1_lacticacid_range = [massc1_lacticacid_range[0]/M_lacticacid, massc1_lacticacid_range[1]/M_lacticacid] # [mol/L] Concentration of lactic acid in Fermentation Broth
c1_h2o_range = [c_p_h2o - (c_p_h2o/c_p_lacticacid)*c1_lacticacid_range[0], c_p_h2o - (c_p_h2o/c_p_lacticacid)*c1_lacticacid_range[1]] # [mol/L] Concentration of water in Fermentation Broth

x1_lacticacid_range = [c1_lacticacid_range[0]/(c1_lacticacid_range[0]+c1_h2o_range[0]), c1_lacticacid_range[1]/(c1_lacticacid_range[1]+c1_h2o_range[1])] # [mol/mol] Mole fraction of lactic acid in Fermentation Broth
# x1_h2o_range = [c1_h2o_range[0]/(c1_lacticacid_range[0]+c1_h2o_range[0]), c1_h2o_range[1]/(c1_lacticacid_range[1]+c1_h2o_range[1])] # [mol/mol] Mole fraction of water in Fermentation Broth

# Load COSMO-solvents dataframe
solvents = pd.read_csv(Setup.solvents_filepath)
solvents = sort_solvents_df(solvents)
# Access the unique row identifier (index) of each row in the DataFrame

# n_solvents = len(solvents)
solvents_ids = solvents.index.tolist()
Design = {
    'temperature': temperature_range,
    'x1_lacticacid': x1_lacticacid_range,
    'solvent': solvents_ids
}

# fullfact = build.full_fact(Design)
levels = [len(temperature_range), len(x1_lacticacid_range), len(solvents_ids)]
reduced_design = pyDOE3.gsd(levels=levels,reduction=9)
reduced_fullfact = pd.DataFrame(reduced_design, columns=['temperature', 'x1_lacticacid', 'solvent'])
reduced_fullfact['temperature'] = reduced_fullfact['temperature'].map({i: temperature_range[i] for i in range(len(temperature_range))})
reduced_fullfact['x1_lacticacid'] = reduced_fullfact['x1_lacticacid'].map({i: x1_lacticacid_range[i] for i in range(len(x1_lacticacid_range))})
reduced_fullfact['solvent'] = reduced_fullfact['solvent'].map({i: solvents_ids[i] for i in range(len(solvents_ids))})

reduced_fullfact = reduced_fullfact.join(solvents['COSMO_name'], on='solvent')

for index, row in reduced_fullfact.iterrows():
    # Extract the values for each parameter
    temperature = row['temperature']
    x1_lacticacid = row['x1_lacticacid']
    solvent = row['COSMO_name']
    

    # print(solvent)
    # # Create the input file for COSMO-RS
    inputfile, filename = gen_LIQEX_inp_file(
    temperature=temperature, 
    x1_lacticacid=x1_lacticacid,
    solvent=solvent,
    ctd_file=Setup.ctd_file,
    cdir=Setup.cdir,
    ldir=Setup.ldir,
    odir=Setup.odir,
    fdir=Setup.fdir,
    output_folder=Setup.inputfile_path
    )
    
    output_file = Setup.odir+"\\"+filename 
    subprocess_str = Setup.cosmotherm_filepath + '"' + inputfile + '"'
    
    print(output_file)
    if os.path.exists(output_file):
        overwrite = input(f"The file '{output_file}' already exists. Do you want to overwrite it? [y/n]: ").strip().lower()
        if overwrite != 'y':
            print("Operation cancelled. The file was not overwritten. returning the existing file path and name.")
        else:
            print("Running COSMO-RS calculation and overwriting the existing file.")
            subprocess.run(subprocess_str, shell=True)
    else:
        print("Running COSMO-RS calculation.")
        subprocess.run(subprocess_str, shell=True)
    

Input file './COSMOtherm/inputfiles/LIQEX_hexane_tc20_x0.0010032800176357408.inp' has been created successfully. returning the file path and name
\\avt.rwth-aachen.de\home$\student\stto01\.AVT-UserConfig\Desktop\COSMOthemOutput\LIQEX_hexane_tc20_x0.0010032800176357408.inp
"C:\Program Files\COSMOlogic\COSMOthermX19\COSMOtherm\BIN-WINDOWS\cosmotherm.exe" "./COSMOtherm/inputfiles/LIQEX_hexane_tc20_x0.0010032800176357408.inp"
Input file './COSMOtherm/inputfiles/LIQEX_4-heptanol_tc20_x0.0010032800176357408.inp' has been created successfully. returning the file path and name
\\avt.rwth-aachen.de\home$\student\stto01\.AVT-UserConfig\Desktop\COSMOthemOutput\LIQEX_4-heptanol_tc20_x0.0010032800176357408.inp
"C:\Program Files\COSMOlogic\COSMOthermX19\COSMOtherm\BIN-WINDOWS\cosmotherm.exe" "./COSMOtherm/inputfiles/LIQEX_4-heptanol_tc20_x0.0010032800176357408.inp"
Input file './COSMOtherm/inputfiles/LIQEX_2-octanol_tc20_x0.0010032800176357408.inp' has been created successfully. returning the file p

In [ ]:
print(Setup.odir+"\\"+filename)

NameError: name 'filename' is not defined

In [ ]:
from doepy import build

# design = build.frac_fact_res(4, gen=['a*b', 'b*c', 'a*c'], resolution=2)
# design = build.frac_fact_res(4, gen=['a*b', 'b*c', 'a*c'], resolution=3)

Param = {'Pressure':[40,55,70],
'Temperature':[290, 320, 350],
'Flow rate':[0.2,0.4],
'Time':[5,8]}

fullfact = build.full_fact(Param)
fullfact


/home/stefan/.cache/pypoetry/virtualenvs/thompsonsampling-XE9c7O9d-py3.12/lib/python3.12/site-packages/doepy/doe_functions.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.iloc[i][j]=r[j][int(df.iloc[i][j])]


,Pressure,Temperature,Flow rate,Time
0,40.0,290.0,0.2,5.0
1,55.0,290.0,0.2,5.0
2,70.0,290.0,0.2,5.0
3,40.0,320.0,0.2,5.0
4,55.0,320.0,0.2,5.0
5,70.0,320.0,0.2,5.0
6,40.0,350.0,0.2,5.0
7,55.0,350.0,0.2,5.0
8,70.0,350.0,0.2,5.0
9,40.0,290.0,0.4,5.0


In [27]:
import pyDOE3

design = pyDOE3.gsd(levels=[2,2,43],reduction=4)
fullfactorialdesign = pyDOE3.fullfact([2,2,43])
print(len(design))
print(len(fullfactorialdesign))
print(design)

44
172
[[ 0  0  0]
 [ 0  0  4]
 [ 0  0  8]
 [ 0  0 12]
 [ 0  0 16]
 [ 0  0 20]
 [ 0  0 24]
 [ 0  0 28]
 [ 0  0 32]
 [ 0  0 36]
 [ 0  0 40]
 [ 0  1  1]
 [ 0  1  5]
 [ 0  1  9]
 [ 0  1 13]
 [ 0  1 17]
 [ 0  1 21]
 [ 0  1 25]
 [ 0  1 29]
 [ 0  1 33]
 [ 0  1 37]
 [ 0  1 41]
 [ 1  0  1]
 [ 1  0  5]
 [ 1  0  9]
 [ 1  0 13]
 [ 1  0 17]
 [ 1  0 21]
 [ 1  0 25]
 [ 1  0 29]
 [ 1  0 33]
 [ 1  0 37]
 [ 1  0 41]
 [ 1  1  2]
 [ 1  1  6]
 [ 1  1 10]
 [ 1  1 14]
 [ 1  1 18]
 [ 1  1 22]
 [ 1  1 26]
 [ 1  1 30]
 [ 1  1 34]
 [ 1  1 38]
 [ 1  1 42]]


In [24]:
build.frac_fact_res(d=Param, res=2)

/home/stefan/.cache/pypoetry/virtualenvs/thompsonsampling-XE9c7O9d-py3.12/lib/python3.12/site-packages/doepy/doe_functions.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.iloc[i][j]=r[j][int(df.iloc[i][j])]


,Pressure,Temperature,Flow rate,Time
0,40.0,290.0,0.2,5.0
1,70.0,290.0,0.2,5.0
2,40.0,350.0,0.2,5.0
3,70.0,350.0,0.2,5.0
4,40.0,290.0,0.4,5.0
5,70.0,290.0,0.4,5.0
6,40.0,350.0,0.4,5.0
7,70.0,350.0,0.4,5.0
8,40.0,290.0,0.2,8.0
9,70.0,290.0,0.2,8.0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

c_p_h2o = np.linspace(0, 55.5, 100) # [mol/L] Concentration of water in Fermentation Broth
c_p_lacticacid = np.linspace(0, 1.209 / M_lacticacid*(10**3), 100) # [mol/L] Concentration of lactic acid in Fermentation Broth
x1_h2o = np.linspace(0, 1, 100) # [mol/L] Concentration of water in Fermentation Broth

c_p_lacticacid = c_p_lacticacid[::-1]

plt.plot(x1_h2o, c_p_h2o, label='Water')
plt.plot(x1_h2o, c_p_lacticacid, label='Lactic Acid')